# ML-enabled SERP competitor analysis (2026 refresh)

Companion notebook for the MLforSEO tutorial **"4 ways to boost your SERP competitor analysis with machine learning"** by Lazarina Stoy (first presented at BrightonSEO, October 2022; refreshed October 2026).

It takes the top organic results for a set of keywords and runs the four analyses from the post:

1. **Categorise your search competitors** by site type and intent (a rule-based starter taxonomy, then an LLM with your own categories), and calculate **share of search** per domain, category and intent.
2. **Extract the prominent entities** from ranking titles and meta descriptions (Google Cloud Natural Language API, with a free spaCy fallback).
3. **Analyse the sentiment** of ranking titles and descriptions (Google Cloud Natural Language API), and filter for your brand.
4. **Analyse language structures with n-grams** to find the title and description patterns Google favours for your keywords.

Everything is exported as tidy CSVs you can drop into the Data Studio template for ML-enabled SERP analysis.

**What you need**

| Step | Needs a key? |
|---|---|
| Load + clean SERP data | No (CSV), or DataForSEO login for a live pull |
| Rule-based categories, share of search | No |
| LLM categories | Anthropic API key (optional) |
| Entities | Google Cloud Natural Language API key, or no key with the spaCy fallback |
| Sentiment | Google Cloud Natural Language API key (optional) |
| N-grams | No |

The notebook runs end to end with no keys at all, on the sample data in this folder. Steps that need a key you haven't set are skipped with a message.

> **Correlation is not causation.** These analyses show you patterns in what ranks. They guide your strategy; they don't replace keyword research or a content strategy.

## 0. Setup

In [ ]:
# In Colab, uncomment the next line (pandas, requests and matplotlib are pre-installed there)
# !pip install -q scikit-learn spacy anthropic && python -m spacy download en_core_web_sm -q

import os, re, json, time
from collections import Counter
from urllib.parse import urlparse

import pandas as pd
import requests
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 80)

In [ ]:
# --- Keys ---------------------------------------------------------------------
# In Colab, store keys in the "Secrets" panel (key icon on the left) and they are
# read below. Locally, set them as environment variables. Leave any you don't
# have empty: those steps are skipped.

def get_secret(name):
    try:
        from google.colab import userdata  # only exists in Colab
        return userdata.get(name)
    except Exception:
        return os.environ.get(name, "")

GOOGLE_NL_API_KEY   = get_secret("GOOGLE_NL_API_KEY")    # Cloud Natural Language API key
ANTHROPIC_API_KEY   = get_secret("ANTHROPIC_API_KEY")    # for LLM categorisation (optional)
DATAFORSEO_LOGIN    = get_secret("DATAFORSEO_LOGIN")     # for a live SERP pull (optional)
DATAFORSEO_PASSWORD = get_secret("DATAFORSEO_PASSWORD")

# --- Your project ---------------------------------------------------------------
BRAND_TERMS = ["mlforseo"]     # used to filter SERP sentiment for your brand
SERP_CSV = "sample_serp_data.csv"  # or your own export: keyword, position, domain, url, title, description

print("Google NL API:", "set" if GOOGLE_NL_API_KEY else "not set (spaCy fallback for entities, sentiment skipped)")
print("Anthropic API:", "set" if ANTHROPIC_API_KEY else "not set (rule-based categories only)")
print("DataForSEO:   ", "set" if DATAFORSEO_LOGIN else "not set (using the CSV)")

## 1. Get the SERP data

You need one row per ranking result: `keyword`, `position`, `domain`, `url`, `title`, `description`. Any rank tracker or SERP export works. The sample file is a real pull of the US top 10 for six machine-learning keywords (DataForSEO, 22 October 2026).

Want a fresh pull? `fetch_serp_dataforseo()` calls DataForSEO's Google Organic **live** endpoint, which takes one keyword per request (about $0.002 per keyword at depth 10 at the time of writing). For hundreds of keywords, use their cheaper standard queue (`task_post` / `task_get`) instead.

In [ ]:
def fetch_serp_dataforseo(keywords, login, password, location_code=2840, language_code="en", depth=10):
    # One request per keyword: the live endpoint ignores extra tasks in the same request.
    rows = []
    for kw in keywords:
        payload = [{"keyword": kw, "location_code": location_code,
                    "language_code": language_code, "depth": depth}]
        r = requests.post("https://api.dataforseo.com/v3/serp/google/organic/live/regular",
                          auth=(login, password), json=payload, timeout=60)
        r.raise_for_status()
        task = r.json()["tasks"][0]
        if task["status_code"] != 20000:
            print(f"{kw}: {task['status_message']}"); continue
        for item in (task["result"][0].get("items") or []):
            if item.get("type") != "organic":
                continue
            rows.append({"keyword": kw, "position": item["rank_group"],
                         "rank_absolute": item["rank_absolute"], "domain": item["domain"],
                         "url": item["url"], "title": item.get("title") or "",
                         "description": item.get("description") or ""})
    return pd.DataFrame(rows)

KEYWORDS = []  # e.g. ["what is machine learning", "machine learning course"]
if KEYWORDS and DATAFORSEO_LOGIN:
    serp = fetch_serp_dataforseo(KEYWORDS, DATAFORSEO_LOGIN, DATAFORSEO_PASSWORD)
else:
    serp = pd.read_csv(SERP_CSV)

print(serp.shape)
serp.head()

In [ ]:
# Clean the snippets: Google prepends a date ("Apr 25, 2025 — ") and DataForSEO
# keeps the "Read more" link text. Both would pollute entities and n-grams.
DATE_PREFIX = re.compile(r"^([A-Z][a-z]{2} \d{1,2}, \d{4})\s+—\s+")

def clean_snippet(text):
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"\s*Read more$", "", text).strip()
    return DATE_PREFIX.sub("", text)

serp["description"] = serp["description"].fillna("")
serp["snippet_date"] = serp["description"].str.extract(DATE_PREFIX)[0]
serp["description"] = serp["description"].map(clean_snippet)
serp["title"] = serp["title"].fillna("")
serp["domain"] = serp["domain"].str.replace(r"^www\.", "", regex=True)
serp[["keyword", "position", "domain", "title", "description", "snippet_date"]].head()

## 2. Categorise your search competitors and calculate share of search

Tag every ranking domain by the **type of site** it is (wiki, courses and learning, news, reviews, forum, video, technology vendor...) and every result by **intent**. Some positions are effectively reserved for big players in each niche, and Google likes a diverse set of results for ambiguous queries, so this shows you who your *real* competition is.

### 2a. A rule-based starter taxonomy (no key needed)
Start with a lookup of domains you already know, plus a few URL and domain patterns. It is fast, free and explainable, and it gives the LLM step good examples to learn from. Edit `KNOWN_DOMAINS` for your niche.

In [ ]:
KNOWN_DOMAINS = {
    "wikipedia.org": "wiki", "investopedia.com": "wiki", "britannica.com": "wiki",
    "coursera.org": "courses & learning", "udemy.com": "courses & learning",
    "datacamp.com": "courses & learning", "codecademy.com": "courses & learning",
    "khanacademy.org": "courses & learning", "geeksforgeeks.org": "courses & learning",
    "reddit.com": "forum & community", "quora.com": "forum & community",
    "youtube.com": "video", "facebook.com": "social media", "linkedin.com": "social media",
    "g2.com": "reviews & comparisons", "capterra.com": "reviews & comparisons",
    "techtarget.com": "news & media", "forbes.com": "news & media",
    "ibm.com": "technology vendor", "aws.amazon.com": "technology vendor",
    "cloud.google.com": "technology vendor", "databricks.com": "technology vendor",
}

def domain_category_rules(domain, url=""):
    d = domain.lower()
    for known, cat in KNOWN_DOMAINS.items():
        if d == known or d.endswith("." + known):
            return cat
    if d.endswith(".edu") or ".ac." in d or d.startswith(("online.", "openlearning.")):
        return "courses & learning"
    if re.search(r"/(best|top)-|/reviews?/|/vs-|compare", url.lower()):
        return "reviews & comparisons"
    if re.search(r"/(blog|news|articles?|insights)/", url.lower()):
        return "blog (unclassified)"
    return "unclassified"

INTENT_PATTERNS = [
    ("transactional", r"\b(buy|price|pricing|discount|coupon|sign up|free trial|enrol|register)\b"),
    ("commercial",    r"\b(best|top \d+|top|vs\.?|versus|review|reviews|compare|comparison|alternatives?)\b"),
    ("navigational",  r"\b(login|log in|official site)\b"),
    ("informational", r"\b(what|how|why|guide|tutorial|explained|definition|examples?|learn|introduction)\b"),
]

def intent_rules(title, description=""):
    text = f"{title} {description}".lower()
    for label, pattern in INTENT_PATTERNS:
        if re.search(pattern, text):
            return label
    return "informational"  # most SERPs default to it; review these

serp["site_category"] = [domain_category_rules(d, u) for d, u in zip(serp["domain"], serp["url"])]
serp["intent"] = [intent_rules(t, s) for t, s in zip(serp["title"], serp["description"])]
serp["site_category"].value_counts()

### 2b. Custom categories with an LLM (optional)
For the domains the rules can't place, give a model your own category list and a few examples. In 2022 I used GPT-4 with Danny Richman's Google Sheets script for this; today I'd use structured outputs, so the model must return valid JSON from your list and nothing else. This uses Claude through the Anthropic SDK; any model with structured outputs works the same way.

You send each domain with one or two of its ranking titles, because "what does this site publish?" is far easier to answer from titles than from a bare domain name.

In [ ]:
CATEGORIES = ["technology vendor", "news & media", "reviews & comparisons", "wiki",
              "courses & learning", "forum & community", "video", "social media",
              "agency & consultancy", "blog & publisher", "other"]
INTENTS = ["informational", "commercial", "transactional", "navigational"]
LLM_MODEL = "claude-opus-5-5"  # cheaper options for big lists: "claude-sonnet-5-5", "claude-haiku-4-5"

def llm_categorise(df, categories=CATEGORIES, model=LLM_MODEL, batch_size=40):
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    domains = (df.groupby("domain")["title"].apply(lambda t: " | ".join(t.head(2)))
                 .reset_index().to_dict("records"))
    examples = [f"{d}: {c}" for d, c in list(KNOWN_DOMAINS.items())[:8]]
    schema = {"type": "object", "additionalProperties": False, "required": ["labels"],
              "properties": {"labels": {"type": "array", "items": {
                  "type": "object", "additionalProperties": False,
                  "required": ["domain", "category", "intent"],
                  "properties": {"domain": {"type": "string"},
                                 "category": {"type": "string", "enum": categories},
                                 "intent": {"type": "string", "enum": INTENTS}}}}}}
    out = []
    for i in range(0, len(domains), batch_size):
        batch = domains[i:i + batch_size]
        listing = "\n".join(f"- {d['domain']} (ranking titles: {d['title']})" for d in batch)
        prompt = ("Classify each website by the type of site it is, based on what it publishes, "
                  "and by the main search intent its ranking titles serve.\n"
                  f"Example site types: {'; '.join(examples)}\n\n"
                  f"Websites:\n{listing}\n\nReturn one label per website.")
        resp = client.messages.create(
            model=model, max_tokens=4000,
            output_config={"format": {"type": "json_schema", "schema": schema},
                           # Haiku 4.5 has no effort parameter (it returns a 400)
                           **({} if model.startswith("claude-haiku-4") else {"effort": "low"})},
            messages=[{"role": "user", "content": prompt}])
        if resp.stop_reason == "refusal":
            print("Batch declined, skipping"); continue
        text = next(b.text for b in resp.content if b.type == "text")
        out.extend(json.loads(text)["labels"])
    return pd.DataFrame(out)

if ANTHROPIC_API_KEY:
    labels = llm_categorise(serp)
    serp = serp.merge(labels.rename(columns={"category": "llm_category", "intent": "llm_intent"}),
                      on="domain", how="left")
    # keep your known domains, let the LLM fill the gaps
    serp["site_category"] = serp["site_category"].where(
        ~serp["site_category"].isin(["unclassified", "blog (unclassified)"]), serp["llm_category"])
else:
    print("No Anthropic key: keeping the rule-based categories.")

### 2c. Share of search per domain, category and intent
Two views:
* **Share of ranking slots**: what percentage of all the top-10 positions a domain (or category) holds. Simple and assumption-free.
* **Click-weighted share**: weights each slot by an estimated click-through rate for its position, so position 1 counts for more than position 9. The CTR curve below is an assumption, not a fact; swap in your own (for example from your Search Console data or a public CTR study).

In [ ]:
CTR_BY_POSITION = {1: .28, 2: .15, 3: .10, 4: .07, 5: .05, 6: .04, 7: .03, 8: .025, 9: .02, 10: .018}
serp["est_ctr"] = serp["position"].map(CTR_BY_POSITION).fillna(0.01)

def share_of_search(df, by):
    g = df.groupby(by).agg(results=("url", "count"), keywords=("keyword", "nunique"),
                           avg_position=("position", "mean"), weighted=("est_ctr", "sum"))
    g["slot_share_%"] = (100 * g["results"] / g["results"].sum()).round(1)
    g["weighted_share_%"] = (100 * g["weighted"] / g["weighted"].sum()).round(1)
    return g.drop(columns="weighted").sort_values(["weighted_share_%", "results"], ascending=False).round(2)

sos_domain = share_of_search(serp, "domain")
sos_category = share_of_search(serp, "site_category")
sos_intent = share_of_search(serp, "intent")
display(sos_category)
sos_domain.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sos_category["weighted_share_%"].sort_values().plot.barh(ax=axes[0], color="#8448ef")
axes[0].set_title("Click-weighted share of search by site type")
sos_category["avg_position"].sort_values(ascending=False).plot.barh(ax=axes[1], color="#251046")
axes[1].set_title("Average position by site type (lower is better)")
plt.tight_layout(); plt.show()

## 3. Identify the prominent entities in ranking results

Which entities keep showing up across the titles and meta descriptions that rank? That can point you to a content direction, and to entities your keyword tool never surfaced.

**Google Cloud Natural Language API.** Use the **v1** `analyzeEntitySentiment` method: it is the only one that still returns **salience** (how central the entity is to the text) together with per-entity **sentiment**. The newer v2 API dropped salience, the Wikipedia URL metadata and the entity-sentiment method. The first 5,000 units a month are free per feature (a unit is 1,000 characters), which covers a lot of titles.

**No key?** The spaCy fallback finds named entities (organisations, products, people, places) for free. It won't give you salience or sentiment, and the small model makes mistakes you'll spot straight away (on the sample data it tags "machine learning" as a person and an organisation). That is exactly why I prefer a purpose-built, knowledge-graph-backed API for this job: treat the spaCy output as a first look, not a finding.

In [ ]:
NL_V1 = "https://language.googleapis.com/v1/documents:{method}?key={key}"
NL_V2 = "https://language.googleapis.com/v2/documents:{method}?key={key}"

def nl_request(url, text, extra=None):
    body = {"document": {"type": "PLAIN_TEXT", "content": text}, "encodingType": "UTF8"}
    body.update(extra or {})
    for attempt in range(4):
        r = requests.post(url, json=body, timeout=30)
        if r.status_code in (429, 500, 503):
            time.sleep(2 ** attempt * 2); continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

def parse_entity_sentiment(resp, row_id, field):
    rows = []
    for e in resp.get("entities", []):
        meta = e.get("metadata", {})
        rows.append({"row_id": row_id, "field": field, "entity": e["name"], "type": e["type"],
                     "salience": e.get("salience"),
                     "sentiment_score": e.get("sentiment", {}).get("score"),
                     "sentiment_magnitude": e.get("sentiment", {}).get("magnitude"),
                     "mentions": len(e.get("mentions", [])),
                     "mid": meta.get("mid"), "wikipedia_url": meta.get("wikipedia_url")})
    return rows

def entities_google(df):
    rows = []
    for i, r in df.iterrows():
        for field in ("title", "description"):
            if r[field]:
                resp = nl_request(NL_V1.format(method="analyzeEntitySentiment", key=GOOGLE_NL_API_KEY), r[field])
                rows += parse_entity_sentiment(resp, i, field)
    return pd.DataFrame(rows)

def entities_spacy(df):
    import spacy
    nlp = spacy.load("en_core_web_sm")
    keep = {"ORG", "PRODUCT", "PERSON", "GPE", "NORP", "WORK_OF_ART", "EVENT", "LOC", "FAC"}
    rows = []
    for i, r in df.iterrows():
        for field in ("title", "description"):
            for ent in nlp(r[field]).ents:
                if ent.label_ in keep:
                    rows.append({"row_id": i, "field": field, "entity": ent.text, "type": ent.label_})
    return pd.DataFrame(rows)

entities = entities_google(serp) if GOOGLE_NL_API_KEY else entities_spacy(serp)
entities = entities.merge(serp[["keyword", "position", "domain"]], left_on="row_id", right_index=True)
entities["entity_norm"] = entities["entity"].str.lower().str.strip()
# drop fragments the small spaCy model sometimes returns (list bullets, cut-off titles)
entities = entities[~entities["entity_norm"].str.contains(r"[:・|]|^\W|\d\s*$", regex=True)]
print(len(entities), "entity mentions found with", "Google NL API" if GOOGLE_NL_API_KEY else "spaCy")

In [ ]:
agg = {"results": ("row_id", "nunique"), "keywords": ("keyword", "nunique"),
       "avg_position": ("position", "mean"),
       "in_title": ("field", lambda f: (f == "title").sum()),
       "in_description": ("field", lambda f: (f == "description").sum())}
if "salience" in entities:
    agg["avg_salience"] = ("salience", "mean")
    agg["avg_sentiment"] = ("sentiment_score", "mean")
agg["type"] = ("type", lambda t: t.mode().iat[0])  # most common type label
entity_summary = (entities.groupby("entity_norm").agg(**agg)
                  .sort_values(["results", "keywords"], ascending=False).round(2))
entity_summary.head(20)

In [ ]:
top = entity_summary.head(15).reset_index()
plt.figure(figsize=(9, 5))
plt.barh(top["entity_norm"][::-1], top["results"][::-1], color="#8448ef")
plt.title("Most common entities across ranking titles and descriptions")
plt.xlabel("Number of ranking results mentioning the entity")
plt.tight_layout(); plt.show()

## 4. Analyse the sentiment of ranking results

Score every title and description for sentiment (from -1, negative, to +1, positive) and magnitude (how strongly it is expressed). Then plot score against position, filter for your brand, and look for the opponents and advocates in your SERPs.

This uses the **v2** `analyzeSentiment` method, which runs on Google's newer model. (Prefer v1 for consistency with older analyses? Swap `NL_V2` for `NL_V1`.) The labels use thresholds you can tune: magnitude tells you whether a near-zero score is genuinely neutral or a mix of strong positive and negative statements.

In [ ]:
def sentiment_label(score, magnitude, pos=0.25, neg=-0.25, mixed_magnitude=1.0):
    if score >= pos:
        return "positive"
    if score <= neg:
        return "negative"
    return "mixed" if magnitude >= mixed_magnitude else "neutral"

def sentiment_google(df):
    out = []
    for i, r in df.iterrows():
        rec = {"row_id": i}
        for field in ("title", "description"):
            if r[field]:
                s = nl_request(NL_V2.format(method="analyzeSentiment", key=GOOGLE_NL_API_KEY), r[field])["documentSentiment"]
                rec[f"{field}_score"], rec[f"{field}_magnitude"] = s.get("score", 0.0), s.get("magnitude", 0.0)
                rec[f"{field}_sentiment"] = sentiment_label(rec[f"{field}_score"], rec[f"{field}_magnitude"])
        out.append(rec)
    return pd.DataFrame(out).set_index("row_id")

if GOOGLE_NL_API_KEY:
    serp = serp.join(sentiment_google(serp))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(serp["title_score"], serp["position"], s=40 + 200 * serp["title_magnitude"],
               alpha=.5, color="#8448ef")
    ax.invert_yaxis(); ax.set_xlabel("Title sentiment score"); ax.set_ylabel("Position")
    ax.set_title("Title sentiment vs position (bubble size = magnitude)")
    plt.show()
    brand = serp[serp[["title", "description"]].apply(
        lambda c: c.str.contains("|".join(BRAND_TERMS), case=False)).any(axis=1)]
    display(brand[["keyword", "position", "domain", "title", "title_sentiment", "description_sentiment"]])
    display(serp.groupby("domain")[["title_score", "description_score"]].mean().sort_values("title_score").head(10))
else:
    print("No Google NL API key: sentiment skipped.")

## 5. Analyse language structures with n-grams

Count the most common **bigrams** (pairs of words) and **trigrams** (triplets) in ranking titles and descriptions. Two passes, because they answer different questions:

* **With stop words kept**, you see title *patterns*: "what is", "how to", "vs", "best ... in 2026". These are the formats Google is rewarding for your keywords.
* **With stop words removed**, you see the *topics*: "machine learning", "artificial intelligence", "supervised learning".

Then split by intent or by position to see which patterns dominate commercial results, or the top three.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(texts, n=(2, 3), stop_words=None, top=15):
    texts = [t for t in texts if t]
    vec = CountVectorizer(ngram_range=n, stop_words=stop_words, lowercase=True,
                          token_pattern=r"(?u)\b[\w+#]+\b")
    X = vec.fit_transform(texts)
    counts = X.sum(axis=0).A1
    docs = (X > 0).sum(axis=0).A1  # in how many titles the n-gram appears
    return (pd.DataFrame({"ngram": vec.get_feature_names_out(), "count": counts, "results": docs})
              .sort_values(["results", "count"], ascending=False).head(top).reset_index(drop=True))

title_patterns = top_ngrams(serp["title"], n=(2, 2))
title_topics = top_ngrams(serp["title"], n=(2, 3), stop_words="english")
desc_topics = top_ngrams(serp["description"], n=(2, 3), stop_words="english")
display(title_patterns.head(10)); display(title_topics.head(10)); desc_topics.head(10)

In [ ]:
serp["position_bucket"] = pd.cut(serp["position"], [0, 3, 10, 100], labels=["top 3", "4-10", "11+"])
by_intent = {i: top_ngrams(g["title"], n=(1, 2), top=8) for i, g in serp.groupby("intent") if len(g) > 2}
by_bucket = {b: top_ngrams(g["title"], n=(2, 2), top=8) for b, g in serp.groupby("position_bucket", observed=True) if len(g) > 2}
for name, table in {**by_intent, **by_bucket}.items():
    print(f"\n== {name}"); print(table.to_string(index=False))

## 6. Export for Data Studio

One tidy CSV per analysis. Upload them to Google Sheets (or BigQuery) and connect them to the Data Studio template for ML-enabled SERP analysis.

In [ ]:
os.makedirs("output", exist_ok=True)
serp.to_csv("output/serp_enriched.csv", index=False)
sos_domain.to_csv("output/share_of_search_by_domain.csv")
sos_category.to_csv("output/share_of_search_by_category.csv")
sos_intent.to_csv("output/share_of_search_by_intent.csv")
entities.to_csv("output/entities_long.csv", index=False)
entity_summary.to_csv("output/entity_summary.csv")
title_patterns.to_csv("output/title_patterns.csv", index=False)
title_topics.to_csv("output/title_topics.csv", index=False)
print(sorted(os.listdir("output")))

try:
    from google.colab import files
    for f in os.listdir("output"):
        files.download(f"output/{f}")
except ImportError:
    pass